# 02 — RNN & LSTM Flickr8k Image Captioning

**Dataset**: Flickr8k (~8.000 gambar, 5 caption per gambar)
**Feature Extractor**: InceptionV3 pretrained (dim=2048)
**Decoder**: SimpleRNN dan LSTM dengan arsitektur pre-inject

---

## Cara Pakai

> **Jalankan semua cell dari atas ke bawah secara berurutan.**

### Alur Kerja
```
[Setup]    gpu-setup → colab-mount → colab-clone → fix-patches → path-setup
              ↓
[Bagian 1] Preprocessing Flickr8k (parse captions, split, vocab)
              ↓
[Bagian 2] Ekstraksi CNN Features (InceptionV3, cached)
              ↓
[Bagian 3] Persiapan Array Training (teacher forcing)
              ↓
[Bagian 4] Training 6 variasi RNN (layer 1/2/3 × hidden 128/512)
              ↓
[Bagian 5] Training 6 variasi LSTM
              ↓
[Bagian 6] Evaluasi & Perbandingan
           ├── BLEU Keras RNN & LSTM (test set)
           ├── Scratch RNN & LSTM inference (load Keras weights)
           ├── Tabel perbandingan BLEU-1/2/3/4
           ├── Contoh kualitatif (GT vs model)
           └── Training curves + BLEU comparison plot
```

### Hasil Disimpan ke Drive
- Bobot RNN  → `MyDrive/weights/rnn/*.h5`
- Bobot LSTM → `MyDrive/weights/lstm/*.h5`
- Hasil JSON → `MyDrive/results/rnn_lstm/*.json`
- Plot       → `MyDrive/results/rnn_lstm/*.png`

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'[GPU] Ditemukan {len(gpus)} GPU: {[g.name for g in gpus]}')
    print(f'[TF]  Versi TensorFlow : {tf.__version__}')
    print(f'[TF]  Built with CUDA  : {tf.test.is_built_with_cuda()}')
else:
    print('[GPU] Tidak ada GPU terdeteksi — menggunakan CPU.')
    try:
        import google.colab
        print('      Di Colab: Runtime → Change runtime type → GPU (T4/L4), lalu Restart session.')
    except ImportError:
        print('      Di lokal: pastikan driver NVIDIA + tensorflow[and-cuda] sudah terinstall.')
    print(f'[TF]  Versi TensorFlow : {tf.__version__}')
    print(f'[TF]  Built with CUDA  : {tf.test.is_built_with_cuda()}')


In [ ]:
import sys

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('[Colab] Google Drive terpasang.')
else:
    print('[Lokal] Tidak di Colab — skip Drive mount.')


In [ ]:
import sys, os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = 'https://github.com/danenftyessir/ChosaHeidan_Tubes-2_IF3270.git'
REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} fetch origin')
        os.system(f'git -C {REPO_DIR} reset --hard origin/main')
    print(f'[Repo] Kode tersedia di {REPO_DIR}/src/')
else:
    print('[Lokal] Skip clone.')


In [ ]:
# ── Reload modul setelah git pull ────────────────────────────────
# Jalankan setelah nb02-clone agar Python memakai versi terbaru modul.
import sys

# 'model_keras' dan 'train' di-import sebagai bare name dari rnn/keras/ dan lstm/keras/
_PREFIXES = ('rnn', 'lstm', 'shared', 'caption_preprocess', 'model_keras', 'train')

_removed = [
    k for k in list(sys.modules.keys())
    if any(k == p or k.startswith(p + '.') for p in _PREFIXES)
]
for k in _removed:
    del sys.modules[k]

print(f'[Reload] {len(_removed)} modul dihapus dari cache:')
for k in sorted(_removed):
    print(f'  - {k}')
if not _removed:
    print('  (tidak ada modul yang di-cache — aman dilanjutkan)')


In [ ]:
# ── Fix relative imports (runtime patch) ─────────────────────────────────────
import os, re

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    _src = os.path.join(REPO_DIR, 'src')

    # 1. Buat __init__.py di semua package dir
    for _d in ['', 'shared', 'cnn', 'cnn/scratch', 'cnn/keras', 'cnn/utils', 'cnn/bonus',
               'lstm', 'lstm/scratch', 'lstm/keras', 'lstm/bonus',
               'rnn', 'rnn/scratch', 'rnn/keras', 'rnn/bonus']:
        _init = os.path.join(_src, _d, '__init__.py')
        if not os.path.exists(_init):
            open(_init, 'w').close()

    def _fix_rel(path, bare_import, make_block):
        with open(path, 'r') as f:
            c = f.read()
        c = re.sub(
            r'(?m)^([ ]*)try:[ ]*\n[ ]*' + re.escape(bare_import.lstrip()) +
            r'[ ]*\n[ ]*except ImportError:[ ]*\n[^\n]*',
            lambda m: m.group(1) + bare_import.lstrip(), c
        )
        c = re.sub(
            r'^([ ]*)' + re.escape(bare_import.lstrip()) + r'$',
            lambda m: make_block(m.group(1)), c, flags=re.MULTILINE
        )
        with open(path, 'w') as f:
            f.write(c)

    # 2. Fix shared/dense.py (dipakai oleh embedding & scratch models)
    _fix_rel(
        os.path.join(_src, 'shared', 'dense.py'),
        'from .activations import get_activation',
        lambda ind: (f'{ind}try:\n'
                     f'{ind}    from .activations import get_activation\n'
                     f'{ind}except ImportError:\n'
                     f'{ind}    from activations import get_activation')
    )

    print('[Fix-02] Patches applied — RNN/LSTM imports OK')
else:
    print('[Fix-02] Lokal — skip patch')


In [ ]:
import os, sys
import numpy as np

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

DRIVE_FLICKR_ID = '11TSyjWcx3mnp6lCXogtNnHrmSyxH5Mew'

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    SRC_DIR  = os.path.join(REPO_DIR, 'src')
    _MYDRIVE = '/content/drive/MyDrive'

    FLICKR8K_DIR     = os.path.join(_MYDRIVE, 'flickr8k')
    RNN_WEIGHTS_DIR  = os.path.join(_MYDRIVE, 'weights', 'rnn')
    LSTM_WEIGHTS_DIR = os.path.join(_MYDRIVE, 'weights', 'lstm')
    VOCAB_DIR        = os.path.join(_MYDRIVE, 'vocab')
    FEATURES_DIR     = os.path.join(_MYDRIVE, 'features')
    RESULTS_DIR      = os.path.join(_MYDRIVE, 'results', 'rnn_lstm')

    # Fallback: Drive API jika folder belum ter-mount
    if not os.path.exists(FLICKR8K_DIR):
        print('[Path] Folder flickr8k tidak ditemukan — pakai Drive API...')
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import io, concurrent.futures

        _svc = build('drive', 'v3')

        def _dl_folder(folder_id, dest, workers=6):
            os.makedirs(dest, exist_ok=True)
            items, page_token = [], None
            while True:
                resp = _svc.files().list(
                    q=f"'{folder_id}' in parents and trashed=false",
                    fields='nextPageToken,files(id,name,mimeType)',
                    pageToken=page_token, pageSize=1000
                ).execute()
                items.extend(resp.get('files', []))
                page_token = resp.get('nextPageToken')
                if not page_token:
                    break
            files = [(i, os.path.join(dest, i['name'])) for i in items
                     if i['mimeType'] != 'application/vnd.google-apps.folder']
            dirs  = [(i, os.path.join(dest, i['name'])) for i in items
                     if i['mimeType'] == 'application/vnd.google-apps.folder']
            def _one(pair):
                item, path = pair
                if os.path.exists(path):
                    return
                req = _svc.files().get_media(fileId=item['id'])
                with io.FileIO(path, 'wb') as fh:
                    dl = MediaIoBaseDownload(fh, req, chunksize=8*1024*1024)
                    done = False
                    while not done:
                        _, done = dl.next_chunk()
            with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
                list(ex.map(_one, files))
            for sub, sub_dest in dirs:
                _dl_folder(sub['id'], sub_dest, workers)

        print('[Drive API] Mengunduh flickr8k ...')
        _dl_folder(DRIVE_FLICKR_ID, FLICKR8K_DIR)

else:
    SRC_DIR = os.path.abspath('.')
    if not os.path.exists(os.path.join(SRC_DIR, 'rnn')):
        SRC_DIR = os.path.join(os.path.abspath('.'), 'src')
    PROJECT_ROOT     = os.path.dirname(SRC_DIR)
    FLICKR8K_DIR     = os.path.join(PROJECT_ROOT, 'data', 'flickr8k')
    RNN_WEIGHTS_DIR  = os.path.join(PROJECT_ROOT, 'weights', 'rnn')
    LSTM_WEIGHTS_DIR = os.path.join(PROJECT_ROOT, 'weights', 'lstm')
    VOCAB_DIR        = os.path.join(PROJECT_ROOT, 'data', 'vocab')
    FEATURES_DIR     = os.path.join(PROJECT_ROOT, 'data', 'features')
    RESULTS_DIR      = os.path.join(PROJECT_ROOT, 'results', 'rnn_lstm')

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, 'shared'))

for d in [RNN_WEIGHTS_DIR, LSTM_WEIGHTS_DIR, VOCAB_DIR, FEATURES_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'[Path] SRC_DIR          : {SRC_DIR}  (exists: {os.path.exists(SRC_DIR)})')
print(f'[Path] FLICKR8K_DIR     : {FLICKR8K_DIR}  (exists: {os.path.exists(FLICKR8K_DIR)})')
print(f'[Path] RNN_WEIGHTS_DIR  : {RNN_WEIGHTS_DIR}')
print(f'[Path] LSTM_WEIGHTS_DIR : {LSTM_WEIGHTS_DIR}')
print(f'[Path] VOCAB_DIR        : {VOCAB_DIR}')
print(f'[Path] FEATURES_DIR     : {FEATURES_DIR}')

if IN_COLAB:
    assert os.path.exists(os.path.join(FLICKR8K_DIR, 'Images')),         f'flickr8k/Images/ tidak ada di: {FLICKR8K_DIR}'
    assert os.path.exists(os.path.join(FLICKR8K_DIR, 'captions.txt')),         f'flickr8k/captions.txt tidak ada di: {FLICKR8K_DIR}'
    print('[Path] Semua path OK')


---

## Bagian 1 — Preprocessing Dataset Flickr8k

Langkah persiapan sebelum training RNN/LSTM:
1. Parse `captions.txt` → dictionary `{image_id: [5 captions]}`
2. Buat train/val/test split (80/10/10) → simpan ke Drive
3. Bangun vocabulary dari training captions (kata dengan freq ≥ 5)

> Hasil disimpan ke `VOCAB_DIR` sehingga tidak perlu diulang di sesi berikutnya.

In [ ]:
import json
import numpy as np
from shared.caption_preprocess import (
    split_captions_file, build_vocabulary,
    save_vocabulary, load_vocabulary,
    get_max_caption_length
)

# ── 1. Parse captions.txt ─────────────────────────────────────────────────────
captions_path = os.path.join(FLICKR8K_DIR, 'captions.txt')
print(f'[B5] Membaca captions: {captions_path}')
captions_dict = split_captions_file(captions_path)
all_image_ids = sorted(captions_dict.keys())
print(f'[B5] Gambar: {len(all_image_ids)} | Caption per gambar: 5')

# ── 2. Buat atau muat train/val/test split ────────────────────────────────────
train_ids_path = os.path.join(VOCAB_DIR, 'train_ids.txt')
val_ids_path   = os.path.join(VOCAB_DIR, 'val_ids.txt')
test_ids_path  = os.path.join(VOCAB_DIR, 'test_ids.txt')

def _load_ids(p):
    return [l.strip() for l in open(p, encoding='utf-8') if l.strip()]

# Muat split HANYA jika file ada DAN tidak kosong
_splits_valid = (
    os.path.exists(train_ids_path) and os.path.getsize(train_ids_path) > 0 and
    os.path.exists(val_ids_path)   and os.path.getsize(val_ids_path)   > 0 and
    os.path.exists(test_ids_path)  and os.path.getsize(test_ids_path)  > 0
)

if _splits_valid:
    train_ids = _load_ids(train_ids_path)
    val_ids   = _load_ids(val_ids_path)
    test_ids  = _load_ids(test_ids_path)
    # Validasi: semua ID harus ada di captions_dict
    _known = set(all_image_ids)
    train_ids = [i for i in train_ids if i in _known]
    val_ids   = [i for i in val_ids   if i in _known]
    test_ids  = [i for i in test_ids  if i in _known]
    if len(train_ids) == 0:
        _splits_valid = False  # Corrupt → recreate

if _splits_valid:
    print(f'[B5] Split dimuat  — train:{len(train_ids)}, val:{len(val_ids)}, test:{len(test_ids)}')
else:
    print('[B5] Split file kosong/tidak valid — membuat split baru...')
    np.random.seed(42)
    perm    = np.random.permutation(len(all_image_ids))
    n_train = int(len(all_image_ids) * 0.80)
    n_val   = int(len(all_image_ids) * 0.10)
    train_ids = [all_image_ids[i] for i in perm[:n_train]]
    val_ids   = [all_image_ids[i] for i in perm[n_train:n_train+n_val]]
    test_ids  = [all_image_ids[i] for i in perm[n_train+n_val:]]
    for path, ids in [(train_ids_path, train_ids),
                      (val_ids_path,   val_ids),
                      (test_ids_path,  test_ids)]:
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(ids))
    print(f'[B5] Split dibuat  — train:{len(train_ids)}, val:{len(val_ids)}, test:{len(test_ids)}')

# ── 3. Bangun atau muat vocabulary ────────────────────────────────────────────
vocab_path    = os.path.join(VOCAB_DIR, 'word2idx.json')
idx2word_path = os.path.join(VOCAB_DIR, 'idx2word.json')

if os.path.exists(vocab_path) and os.path.getsize(vocab_path) > 0:
    word2idx = load_vocabulary(vocab_path)
    with open(idx2word_path, encoding='utf-8') as f:
        idx2word = {int(k): v for k, v in json.load(f).items()}
    print(f'[B5] Vocab dimuat  — {len(word2idx)} kata')
else:
    train_caps = {img: captions_dict[img] for img in train_ids if img in captions_dict}
    word2idx, idx2word, _ = build_vocabulary(train_caps, min_freq=5)
    save_vocabulary(word2idx, vocab_path)
    with open(idx2word_path, 'w', encoding='utf-8') as f:
        json.dump({str(k): v for k, v in idx2word.items()}, f)
    print(f'[B5] Vocab dibangun — {len(word2idx)} kata (min_freq=5)')

VOCAB_SIZE = len(word2idx)
MAX_LENGTH = min(get_max_caption_length(captions_dict) + 2, 40)  # +2 start/end

print(f'\n[B5] VOCAB_SIZE = {VOCAB_SIZE}')
print(f'[B5] MAX_LENGTH = {MAX_LENGTH}  (termasuk <start> dan <end>)')


## Bagian 2 — Ekstraksi CNN Features (InceptionV3)

Ekstrak feature vector (dim=2048) dari seluruh gambar Flickr8k menggunakan InceptionV3 pretrained.

> **Hanya berjalan sekali** — hasil disimpan ke `FEATURES_DIR/flickr8k_inception.npy`.
> Sesi berikutnya langsung load dari file. Estimasi waktu ekstraksi: ~10–15 menit di T4/L4 GPU.

In [ ]:
import json, os, sys

# Guard: pastikan SRC_DIR ada di sys.path sebelum import shared
if not any(os.path.isdir(os.path.join(p, 'shared')) for p in sys.path):
    for _cand in [
        '/content/ChosaHeidan_Tubes-2_IF3270/src',
        os.path.join(os.path.abspath('.'), 'src'),
        os.path.abspath('.'),
    ]:
        if os.path.isdir(os.path.join(_cand, 'shared')):
            sys.path.insert(0, _cand)
            print(f'[B6] sys.path patched → {_cand}')
            break

from shared.feature_extract import (
    extract_features_inceptionv3, save_cnn_features, load_cnn_features
)

FEATURES_PATH  = os.path.join(FEATURES_DIR, 'flickr8k_inception.npy')
FEAT_IDS_PATH  = os.path.join(FEATURES_DIR, 'flickr8k_image_ids.json')
IMAGE_DIR      = os.path.join(FLICKR8K_DIR, 'Images')

if os.path.exists(FEATURES_PATH) and os.path.exists(FEAT_IDS_PATH):
    print('[B6] Features sudah ada — skip ekstraksi, langsung muat...')
    raw_features   = load_cnn_features(FEATURES_PATH)
    with open(FEAT_IDS_PATH, encoding='utf-8') as f:
        feat_image_ids = json.load(f)
else:
    # ── Recover all_image_ids (3 lapis fallback) ──────────────────────────────
    if 'all_image_ids' not in globals() or len(all_image_ids) == 0:
        # Lapis 1: coba muat dari split files
        _split_files = [
            os.path.join(VOCAB_DIR, 'train_ids.txt'),
            os.path.join(VOCAB_DIR, 'val_ids.txt'),
            os.path.join(VOCAB_DIR, 'test_ids.txt'),
        ]
        if all(os.path.exists(p) for p in _split_files):
            all_image_ids = []
            for _p in _split_files:
                all_image_ids += [l.strip() for l in open(_p, encoding='utf-8') if l.strip()]

        # Lapis 2: split files kosong/tidak ada → parse langsung captions.txt
        if len(all_image_ids) == 0:
            _captions_txt = os.path.join(FLICKR8K_DIR, 'captions.txt')
            if os.path.exists(_captions_txt):
                from shared.caption_preprocess import split_captions_file
                _caps = split_captions_file(_captions_txt)
                all_image_ids = sorted(_caps.keys())
                print(f'[B6] all_image_ids di-recover dari captions.txt: {len(all_image_ids)} gambar')
            else:
                raise RuntimeError(
                    f'captions.txt tidak ditemukan di: {FLICKR8K_DIR}\n'
                    'Pastikan struktur Drive: flickr8k/captions.txt dan flickr8k/Images/'
                )

        if len(all_image_ids) == 0:
            raise RuntimeError('all_image_ids masih kosong setelah semua fallback. Cek isi captions.txt.')

        print(f'[B6] all_image_ids tersedia: {len(all_image_ids)} gambar')

    print(f'[B6] Ekstraksi InceptionV3 untuk {len(all_image_ids)} gambar...')
    print(f'     Image dir : {IMAGE_DIR}')
    print(f'     Output    : {FEATURES_PATH}')
    raw_features   = extract_features_inceptionv3(
        all_image_ids, image_dir=IMAGE_DIR, batch_size=32, verbose=True
    )
    feat_image_ids = all_image_ids
    save_cnn_features(raw_features, FEATURES_PATH)
    with open(FEAT_IDS_PATH, 'w', encoding='utf-8') as f:
        json.dump(feat_image_ids, f)

# Buat lookup dict: image_id -> feature vector
features_dict = {img_id: raw_features[i]
                 for i, img_id in enumerate(feat_image_ids)}
FEATURE_DIM   = raw_features.shape[1]

print(f'\n[B6] features_dict : {len(features_dict)} gambar')
print(f'[B6] FEATURE_DIM   : {FEATURE_DIM}')


## Bagian 3 — Persiapan Array Training (Teacher Forcing)

Konversi caption teks → array numerik untuk training RNN/LSTM.

**Format teacher forcing:**
```
caption : [<start>, w1, w2, w3, <end>, <pad>, ...]
input   : [<start>, w1, w2, w3, <end>, <pad>]   ← seq_train
target  : [w1,      w2, w3, <end>, <pad>, <pad>] ← lbl_train
```
Setiap gambar punya 5 caption → tiap gambar menghasilkan 5 baris training.

In [ ]:
from shared.caption_preprocess import prepare_training_data
import numpy as np

def build_split_arrays(image_ids, captions_dict, features_dict, word2idx, max_length):
    """
    Buat (cnn_features, input_seqs, target_seqs, img_ids) untuk satu split.
    Setiap gambar × 5 caption = N baris array.
    Returns img_ids agar bisa dipakai untuk qualitative display.
    """
    matched_ids, full_seqs = prepare_training_data(
        captions_dict, image_ids, word2idx,
        max_length=max_length, add_start=True, add_end=True
    )
    valid = [(i, img_id) for i, img_id in enumerate(matched_ids)
             if img_id in features_dict]
    if not valid:
        raise ValueError('Tidak ada gambar yang cocok dengan features_dict!')

    idxs    = [v[0] for v in valid]
    img_ids = [v[1] for v in valid]

    seqs      = full_seqs[idxs]
    cnn_feats = np.array([features_dict[i] for i in img_ids])

    input_seqs  = seqs[:, :-1]
    target_seqs = seqs[:, 1:]

    return cnn_feats, input_seqs, target_seqs, img_ids

print('[B3] Membangun array training (teacher forcing)...')
cnn_train, seq_train, lbl_train, train_img_ids = build_split_arrays(
    train_ids, captions_dict, features_dict, word2idx, MAX_LENGTH)
cnn_val,   seq_val,   lbl_val,   val_img_ids   = build_split_arrays(
    val_ids,   captions_dict, features_dict, word2idx, MAX_LENGTH)
cnn_test,  seq_test,  lbl_test,  test_img_ids  = build_split_arrays(
    test_ids,  captions_dict, features_dict, word2idx, MAX_LENGTH)

SEQ_LENGTH = seq_train.shape[1]   # MAX_LENGTH - 1
IMAGE_DIR  = os.path.join(FLICKR8K_DIR, 'Images')

print(f'\n{"Split":<8} | {"Pasang":>8} | {"cnn_shape":>15} | {"seq_shape":>15}')
print('-' * 55)
for name, cnn, seq in [('Train', cnn_train, seq_train),
                        ('Val',   cnn_val,   seq_val),
                        ('Test',  cnn_test,  seq_test)]:
    print(f'{name:<8} | {cnn.shape[0]:>8,} | {str(cnn.shape):>15} | {str(seq.shape):>15}')

print(f'\n[B3] SEQ_LENGTH = {SEQ_LENGTH}')
print(f'[B3] VOCAB_SIZE = {VOCAB_SIZE}, FEATURE_DIM = {FEATURE_DIM}')
print('[B3] Siap untuk training RNN & LSTM!')


---

## Bagian 4 — Training RNN

Train **6 variasi RNN** (SimpleRNN decoder, arsitektur preinject):

| Hyperparameter | Nilai |
|---|---|
| Jumlah layer | 1, 2, 3 |
| Hidden dim | 128, 512 |

**Total**: 3 × 2 = **6 model RNN**

> Bobot terbaik (best val_loss) setiap model disimpan ke `weights/rnn/`.
> Early stopping patience = 5. Estimasi waktu: ~20–40 menit di L4 GPU.

In [ ]:
import json as _json, os as _os
from rnn.keras.train import train_with_variations as rnn_train_variations

_rnn_cmp_json = _os.path.join(RESULTS_DIR, 'rnn_comparison.json')

if _os.path.exists(_rnn_cmp_json):
    with open(_rnn_cmp_json) as _f:
        rnn_comparison = _json.load(_f)
    rnn_results = None
    print(f'[B8] Cache ditemukan — skip training, {len(rnn_comparison)} model dimuat dari JSON.')
else:
    print('[B8] Mulai training 6 variasi RNN...')
    print(f'     cnn_train : {cnn_train.shape}')
    print(f'     seq_train : {seq_train.shape}')
    print(f'     VOCAB_SIZE: {VOCAB_SIZE}, FEATURE_DIM: {FEATURE_DIM}')
    print(f'     SEQ_LENGTH: {SEQ_LENGTH}')
    rnn_results = rnn_train_variations(
        cnn_features_train=cnn_train,
        train_seq=seq_train,
        train_labels=lbl_train,
        val_cnn_features=cnn_val,
        val_seq=seq_val,
        val_labels=lbl_val,
        vocab_size=VOCAB_SIZE,
        embed_dim=256,
        layer_variations=[1, 2, 3],
        hidden_variations=[128, 512],
        feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH,
        epochs=30,
        batch_size=64,
        lr=0.001,
        weights_dir=RNN_WEIGHTS_DIR,
        architecture='preinject',
        dropout=0.3,
        verbose=1,
    )
    rnn_comparison = None
    print(f'[B8] Selesai — {len(rnn_results)} model RNN dilatih.')


In [ ]:
from rnn.keras.train import compare_training_results as rnn_compare

# Hitung comparison jika belum ada (hasil training baru)
if rnn_comparison is None:
    rnn_comparison = rnn_compare(
        rnn_results,
        save_path=os.path.join(RESULTS_DIR, 'rnn_comparison.json'),
    )

rnn_ranked = sorted(
    [(k, v) for k, v in rnn_comparison.items()
     if v.get('best_val_loss') is not None],
    key=lambda x: x[1]['best_val_loss'],
)

print(f'\n{"="*65}')
print('  RANKING RNN — Best Val Loss (lower is better)')
print(f'{"="*65}')
for i, (name, data) in enumerate(rnn_ranked, 1):
    mark = ' <-- BEST' if i == 1 else ''
    print(
        f'  {i:2d}. {name:<35} '
        f'val_loss={data["best_val_loss"]:.4f}  '
        f'ep={data["epochs_trained"]}{mark}'
    )

if rnn_ranked:
    BEST_RNN_NAME = rnn_ranked[0][0]
    # Load model dari memory (fresh training) atau dari .h5 (resume)
    if rnn_results is not None and rnn_results.get(BEST_RNN_NAME, {}).get('model') is not None:
        BEST_RNN_MODEL = rnn_results[BEST_RNN_NAME]['model']
    else:
        from rnn.keras.train import build_model as _rnn_build_model
        _cfg = rnn_comparison[BEST_RNN_NAME]
        _best_weights_h5 = os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.weights.h5')
        BEST_RNN_MODEL = _rnn_build_model(
            vocab_size=VOCAB_SIZE, embed_dim=256,
            hidden_dim=_cfg['hidden_dim'], num_layers=_cfg['num_layers'],
            feature_dim=FEATURE_DIM, seq_max_length=SEQ_LENGTH,
        )
        BEST_RNN_MODEL.load_weights(_best_weights_h5)
        print(f'[B8] Model di-load dari: {_best_weights_h5}')
    print(f'\n[B8] Best RNN model : {BEST_RNN_NAME}')
    print(f'[B8] Bobot tersimpan: {RNN_WEIGHTS_DIR}/{BEST_RNN_NAME}_best.weights.h5')


### Bagian 4-Bonus — Pre-Inject vs Init-Inject Architecture (RNN)

Dua arsitektur injeksi CNN feature ke dalam decoder RNN:
- **Pre-Inject**: feature CNN dikonversi ke embed_dim dan menjadi token pertama (x_{-1})
- **Init-Inject**: RNN memproses token saja, lalu [h_T ; CNN_projected] → Dense → softmax

> Perbandingan menggunakan scratch model dengan bobot acak (demonstrasi arsitektur).

In [ ]:
from rnn.bonus.bonus_init_inject import (
    RNNInitInject, build_initinject_from_config, compare_preinject_vs_initinject
)

print('[Bonus-InitInject-RNN] Membangun model Init-Inject RNN...')

best_rnn_cfg = rnn_results[BEST_RNN_NAME]['config']

# Build Init-Inject scratch model
rnn_initinject = build_initinject_from_config(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_rnn_cfg['embed_dim'],
    hidden_dim=best_rnn_cfg['hidden_dim'],
    num_layers=best_rnn_cfg['num_layers'],
    feature_dim=FEATURE_DIM,
)
rnn_initinject.build()
print(f'  RNNInitInject dibangun: hidden_dim={best_rnn_cfg["hidden_dim"]}, '
      f'num_layers={best_rnn_cfg["num_layers"]}')

# Pre-inject scratch (sudah dibangun di Bagian 6)
print('\n[Bonus-InitInject-RNN] Perbandingan Pre-Inject vs Init-Inject pada sample:')
compare_preinject_vs_initinject(
    preinject_model=rnn_scratch,
    initinject_model=rnn_initinject,
    cnn_features=cnn_test[:5],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    n_samples=5,
)


---

## Bagian 5 — Training LSTM

Train **6 variasi LSTM** (arsitektur identik dengan RNN, LSTM cell menggantikan SimpleRNN):

| Hyperparameter | Nilai |
|---|---|
| Jumlah layer | 1, 2, 3 |
| Hidden dim | 128, 512 |

**Total**: 3 × 2 = **6 model LSTM**

> LSTM memiliki gating mechanism (forget, input, output gate) sehingga lebih baik
> menangani long-range dependencies dibanding SimpleRNN — namun lebih lambat saat training.
> Bobot terbaik disimpan ke `weights/lstm/`. Estimasi waktu: ~30–60 menit di L4 GPU.

In [ ]:
import json as _json, os as _os
from lstm.keras.train import train_with_variations as lstm_train_variations

_lstm_cmp_json = _os.path.join(RESULTS_DIR, 'lstm_comparison.json')

if _os.path.exists(_lstm_cmp_json):
    with open(_lstm_cmp_json) as _f:
        lstm_comparison = _json.load(_f)
    lstm_results = None
    print(f'[B9] Cache ditemukan — skip training, {len(lstm_comparison)} model dimuat dari JSON.')
else:
    print('[B9] Mulai training 6 variasi LSTM...')
    lstm_results = lstm_train_variations(
        cnn_features_train=cnn_train,
        train_seq=seq_train,
        train_labels=lbl_train,
        val_cnn_features=cnn_val,
        val_seq=seq_val,
        val_labels=lbl_val,
        vocab_size=VOCAB_SIZE,
        embed_dim=256,
        layer_variations=[1, 2, 3],
        hidden_variations=[128, 512],
        feature_dim=FEATURE_DIM,
        seq_max_length=SEQ_LENGTH,
        epochs=30,
        batch_size=64,
        lr=0.001,
        weights_dir=LSTM_WEIGHTS_DIR,
        architecture='preinject',
        dropout=0.3,
        verbose=1,
    )
    lstm_comparison = None
    print(f'[B9] Selesai — {len(lstm_results)} model LSTM dilatih.')


In [ ]:
from lstm.keras.train import compare_training_results as lstm_compare

# Hitung comparison jika belum ada (hasil training baru)
if lstm_comparison is None:
    lstm_comparison = lstm_compare(
        lstm_results,
        save_path=os.path.join(RESULTS_DIR, 'lstm_comparison.json'),
    )

lstm_ranked = sorted(
    [(k, v) for k, v in lstm_comparison.items()
     if v.get('best_val_loss') is not None],
    key=lambda x: x[1]['best_val_loss'],
)

print(f'\n{"="*65}')
print('  RANKING LSTM — Best Val Loss (lower is better)')
print(f'{"="*65}')
for i, (name, data) in enumerate(lstm_ranked, 1):
    mark = ' <-- BEST' if i == 1 else ''
    print(
        f'  {i:2d}. {name:<35} '
        f'val_loss={data["best_val_loss"]:.4f}  '
        f'ep={data["epochs_trained"]}{mark}'
    )

if lstm_ranked:
    BEST_LSTM_NAME = lstm_ranked[0][0]
    # Load model dari memory (fresh training) atau dari .h5 (resume)
    if lstm_results is not None and lstm_results.get(BEST_LSTM_NAME, {}).get('model') is not None:
        BEST_LSTM_MODEL = lstm_results[BEST_LSTM_NAME]['model']
    else:
        from lstm.keras.train import build_model as _lstm_build_model
        _cfg = lstm_comparison[BEST_LSTM_NAME]
        _best_weights_h5 = os.path.join(LSTM_WEIGHTS_DIR, f'{BEST_LSTM_NAME}_best.weights.h5')
        BEST_LSTM_MODEL = _lstm_build_model(
            vocab_size=VOCAB_SIZE, embed_dim=256,
            hidden_dim=_cfg['hidden_dim'], num_layers=_cfg['num_layers'],
            feature_dim=FEATURE_DIM, seq_max_length=SEQ_LENGTH,
        )
        BEST_LSTM_MODEL.load_weights(_best_weights_h5)
        print(f'[B9] Model di-load dari: {_best_weights_h5}')
    print(f'\n[B9] Best LSTM model : {BEST_LSTM_NAME}')
    print(f'[B9] Bobot tersimpan : {LSTM_WEIGHTS_DIR}/{BEST_LSTM_NAME}_best.weights.h5')


### Bagian 5-Bonus — Pre-Inject vs Init-Inject Architecture (LSTM)

Sama seperti RNN, LSTM juga mendukung dua arsitektur injeksi CNN feature.

In [ ]:
from lstm.bonus.bonus_init_inject import (
    LSTMInitInject, build_lstm_initinject_from_config,
    compare_lstm_preinject_vs_initinject
)

print('[Bonus-InitInject-LSTM] Membangun model Init-Inject LSTM...')

best_lstm_cfg = lstm_results[BEST_LSTM_NAME]['config']

lstm_initinject = build_lstm_initinject_from_config(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_lstm_cfg['embed_dim'],
    hidden_dim=best_lstm_cfg['hidden_dim'],
    num_layers=best_lstm_cfg['num_layers'],
    feature_dim=FEATURE_DIM,
)
lstm_initinject.build()
print(f'  LSTMInitInject dibangun: hidden_dim={best_lstm_cfg["hidden_dim"]}, '
      f'num_layers={best_lstm_cfg["num_layers"]}')

print('\n[Bonus-InitInject-LSTM] Perbandingan Pre-Inject vs Init-Inject pada sample:')
compare_lstm_preinject_vs_initinject(
    preinject_model=lstm_scratch,
    initinject_model=lstm_initinject,
    cnn_features=cnn_test[:5],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    n_samples=5,
)


## Bagian 6 — Evaluasi & Perbandingan RNN vs LSTM

Evaluasi dilakukan dalam beberapa langkah:
1. **Ground truth** — dekode `lbl_test` ke caption string
2. **Keras RNN & LSTM** — evaluasi model Keras terbaik di test set (BLEU-1/2/3/4)
3. **Scratch RNN & LSTM** — muat bobot Keras ke implementasi NumPy, jalankan greedy decode
4. **Tabel perbandingan** — BLEU-1/2/3/4 untuk semua model
5. **Contoh kualitatif** — caption Ground Truth vs Keras-RNN vs Keras-LSTM vs Scratch-RNN vs Scratch-LSTM
6. **Training curves** — plot loss per epoch untuk model terbaik

In [ ]:
# ── 6a. Dekode target sequences → ground truth caption strings ─────────────────
def decode_target_seqs(target_seqs, idx2word):
    """Decode lbl_test rows → list of caption string."""
    END_WORD = '<end>'
    SKIP = {'<start>', '<end>', '<pad>', ''}
    captions = []
    for seq in target_seqs:
        words = []
        for t in seq:
            w = idx2word.get(int(t), '')
            if w == END_WORD:
                break
            if w not in SKIP:
                words.append(w)
        captions.append(' '.join(words))
    return captions

gt_captions_test = decode_target_seqs(lbl_test, idx2word)
print(f'[B6a] Ground truth captions (test) : {len(gt_captions_test)}')
print(f'      Contoh [0] : {gt_captions_test[0]}')


In [ ]:
from rnn.keras.evaluate import evaluate_model as rnn_evaluate_keras

print('\n[B6b] Evaluasi Keras RNN terbaik pada test set...')
rnn_metrics, rnn_preds = rnn_evaluate_keras(
    model=BEST_RNN_MODEL,
    cnn_features=cnn_test,
    gt_captions=gt_captions_test,
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=64,
    verbose=True,
)
print(f'\n  RNN  BLEU-1: {rnn_metrics.get("bleu1",0):.4f}')
print(f'  RNN  BLEU-2: {rnn_metrics.get("bleu2",0):.4f}')
print(f'  RNN  BLEU-3: {rnn_metrics.get("bleu3",0):.4f}')
print(f'  RNN  BLEU-4: {rnn_metrics.get("bleu4",0):.4f}')


In [ ]:
from lstm.keras.evaluate import evaluate_model as lstm_evaluate_keras

print('\n[B6c] Evaluasi Keras LSTM terbaik pada test set...')
lstm_metrics, lstm_preds = lstm_evaluate_keras(
    model=BEST_LSTM_MODEL,
    cnn_features=cnn_test,
    gt_captions=gt_captions_test,
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=64,
    verbose=True,
)
print(f'\n  LSTM BLEU-1: {lstm_metrics.get("bleu1",0):.4f}')
print(f'  LSTM BLEU-2: {lstm_metrics.get("bleu2",0):.4f}')
print(f'  LSTM BLEU-3: {lstm_metrics.get("bleu3",0):.4f}')
print(f'  LSTM BLEU-4: {lstm_metrics.get("bleu4",0):.4f}')


In [ ]:
from rnn.scratch.model_scratch import RNNScratch

print('\n[B6d] Scratch RNN — muat bobot Keras .h5 ke implementasi NumPy...')

best_rnn_cfg = rnn_results[BEST_RNN_NAME]['config']
rnn_scratch = RNNScratch(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_rnn_cfg['embed_dim'],
    hidden_dim=best_rnn_cfg['hidden_dim'],
    num_layers=best_rnn_cfg['num_layers'],
    feature_dim=FEATURE_DIM,
)
rnn_scratch.build()

best_rnn_h5 = os.path.join(RNN_WEIGHTS_DIR, f'{BEST_RNN_NAME}_best.weights.h5')
if os.path.exists(best_rnn_h5):
    rnn_scratch.load_weights_from_h5(best_rnn_h5)
    print(f'  Bobot dimuat dari: {best_rnn_h5}')
else:
    print(f'  [WARN] File tidak ditemukan: {best_rnn_h5}')
    print('         Jalankan Bagian 4 terlebih dahulu.')

# Greedy decode 5 sample dari test set
N_SAMPLE = 5
sample_feats = cnn_test[:N_SAMPLE]
scratch_rnn_caps = rnn_scratch.greedy_decode_batch(
    sample_feats, idx2word, max_length=SEQ_LENGTH
)
print(f'\n[B6d] Scratch RNN greedy decode ({N_SAMPLE} sample):')
for i, cap in enumerate(scratch_rnn_caps):
    print(f'  [{i+1}] {cap}')


In [ ]:
from lstm.scratch.model_scratch import LSTMScratch

print('\n[B6e] Scratch LSTM — muat bobot Keras .h5 ke implementasi NumPy...')

best_lstm_cfg = lstm_results[BEST_LSTM_NAME]['config']
lstm_scratch = LSTMScratch(
    vocab_size=VOCAB_SIZE,
    embed_dim=best_lstm_cfg['embed_dim'],
    hidden_dim=best_lstm_cfg['hidden_dim'],
    num_layers=best_lstm_cfg['num_layers'],
    feature_dim=FEATURE_DIM,
)
lstm_scratch.build()

best_lstm_h5 = os.path.join(LSTM_WEIGHTS_DIR, f'{BEST_LSTM_NAME}_best.weights.h5')
if os.path.exists(best_lstm_h5):
    lstm_scratch.load_weights_from_h5(best_lstm_h5)
    print(f'  Bobot dimuat dari: {best_lstm_h5}')
else:
    print(f'  [WARN] File tidak ditemukan: {best_lstm_h5}')
    print('         Jalankan Bagian 5 terlebih dahulu.')

scratch_lstm_caps = lstm_scratch.greedy_decode_batch(
    sample_feats, idx2word, max_length=SEQ_LENGTH
)
print(f'\n[B6e] Scratch LSTM greedy decode ({N_SAMPLE} sample):')
for i, cap in enumerate(scratch_lstm_caps):
    print(f'  [{i+1}] {cap}')


In [ ]:
import json as _json

METRICS = ['bleu1', 'bleu2', 'bleu3', 'bleu4']

print(f'\n{"="*70}')
print('  PERBANDINGAN BLEU — Keras RNN vs Keras LSTM — Test Set')
print(f'{"="*70}')
print(f'  {"Metric":<15} {"RNN":>12} {"LSTM":>12} {"Winner":>10}')
print(f'  {"-"*52}')

for m in METRICS:
    rv = rnn_metrics.get(m, 0.0)
    lv = lstm_metrics.get(m, 0.0)
    winner = 'LSTM' if lv > rv else ('RNN' if rv > lv else 'tie')
    print(f'  {m.upper():<15} {rv:>12.4f} {lv:>12.4f} {winner:>10}')

avg_rnn  = sum(rnn_metrics.get(m, 0)  for m in METRICS) / 4
avg_lstm = sum(lstm_metrics.get(m, 0) for m in METRICS) / 4
winner   = 'LSTM' if avg_lstm > avg_rnn else 'RNN'
print(f'  {"-"*52}')
print(f'  {"Avg BLEU":<15} {avg_rnn:>12.4f} {avg_lstm:>12.4f} {winner:>10}')
print(f'{"="*70}')

# Simpan metrik
metrics_out = {
    'rnn':  {'model': BEST_RNN_NAME,  **rnn_metrics},
    'lstm': {'model': BEST_LSTM_NAME, **lstm_metrics},
}
eval_path = os.path.join(RESULTS_DIR, 'rnn_lstm_eval.json')
with open(eval_path, 'w') as f:
    _json.dump(metrics_out, f, indent=2)
print(f'\n[B6f] Metrik disimpan ke: {eval_path}')


In [ ]:
# ── Contoh kualitatif: GT vs Keras-RNN vs Keras-LSTM vs Scratch-RNN vs Scratch-LSTM
from shared.plot_utils import plot_caption_samples

N_SHOW = 5
sample_img_ids   = test_img_ids[:N_SHOW]
sample_img_paths = [os.path.join(IMAGE_DIR, img_id) for img_id in sample_img_ids]

print(f'{"="*70}')
print(f'  CONTOH CAPTION ({N_SHOW} sample dari test set)')
print(f'{"="*70}')

for idx in range(N_SHOW):
    print(f'\n  [{idx+1}] Gambar: {sample_img_ids[idx]}')
    print(f'       GT           : {gt_captions_test[idx]}')
    print(f'       Keras RNN    : {rnn_preds[idx]}')
    print(f'       Keras LSTM   : {lstm_preds[idx]}')
    print(f'       Scratch RNN  : {scratch_rnn_caps[idx] if idx < len(scratch_rnn_caps) else "-"}')
    print(f'       Scratch LSTM : {scratch_lstm_caps[idx] if idx < len(scratch_lstm_caps) else "-"}')

# Plot qualitative dengan gambar asli (GT vs Keras RNN)
plot_caption_samples(
    image_paths=sample_img_paths,
    gt_captions=gt_captions_test[:N_SHOW],
    pred_captions=rnn_preds[:N_SHOW],
    save_path=os.path.join(RESULTS_DIR, 'qualitative_rnn.png'),
    n=N_SHOW,
)
# Plot LSTM captions
plot_caption_samples(
    image_paths=sample_img_paths,
    gt_captions=gt_captions_test[:N_SHOW],
    pred_captions=lstm_preds[:N_SHOW],
    save_path=os.path.join(RESULTS_DIR, 'qualitative_lstm.png'),
    n=N_SHOW,
)


In [ ]:
from shared.plot_utils import plot_training_history, plot_bleu_comparison

# ── Training curves untuk model terbaik RNN dan LSTM ──────────────────────────
for label, results, name in [
    ('RNN',  rnn_results,  BEST_RNN_NAME),
    ('LSTM', lstm_results, BEST_LSTM_NAME),
]:
    history = results[name]['history'].history
    # plot_training_history expects {'train_loss': [...], 'val_loss': [...]}
    hist_dict = {
        'train_loss': history.get('loss', []),
        'val_loss':   history.get('val_loss', []),
    }
    plot_training_history(
        history=hist_dict,
        title=f'{label} Training — {name}',
        save_path=os.path.join(RESULTS_DIR, f'training_curve_{label.lower()}.png'),
    )

# ── BLEU comparison bar chart ─────────────────────────────────────────────────
bleu_compare = {
    'RNN':  rnn_metrics,
    'LSTM': lstm_metrics,
}
plot_bleu_comparison(
    results_dict=bleu_compare,
    title='BLEU Score — RNN vs LSTM (Test Set)',
    save_path=os.path.join(RESULTS_DIR, 'bleu_comparison.png'),
)
print('[B6h] Training curves dan BLEU comparison plot selesai.')


## Bagian 6-Bonus-1 — Beam Search vs Greedy Decoding

**Beam search** (k=5) menjelajahi k kandidat caption secara bersamaan di setiap langkah,
sehingga menghasilkan caption yang secara global lebih baik dibanding greedy (k=1).

Perbandingan dilakukan pada **scratch RNN** dan **scratch LSTM** yang sudah dilatih.

In [ ]:
from rnn.bonus.bonus_beam_search import (
    beam_search, compare_beam_vs_greedy, beam_search_with_length_penalty
)
from lstm.bonus.bonus_beam_search import (
    beam_search_lstm, compare_lstm_beam_vs_greedy,
    beam_search_lstm_with_length_penalty
)

N_BEAM = 5   # jumlah sample untuk perbandingan

# ── RNN: Beam Search vs Greedy ────────────────────────────────────────────────
print('[Bonus-Beam] RNN: Beam Search (k=5) vs Greedy')
print('='*65)
rnn_beam_results = compare_beam_vs_greedy(
    model=rnn_scratch,
    cnn_features=cnn_test[:N_BEAM],
    idx2word=idx2word,
    k=5,
    max_length=SEQ_LENGTH,
    n_samples=N_BEAM,
)

# RNN dengan length penalty
print('\n[Bonus-Beam] RNN: Beam Search dengan Length Penalty (alpha=0.6)')
for i, feat in enumerate(cnn_test[:3]):
    cap_lp = beam_search_with_length_penalty(
        model=rnn_scratch,
        cnn_feature=feat,
        idx2word=idx2word,
        k=5,
        max_length=SEQ_LENGTH,
        alpha=0.6,
    )
    print(f'  [{i+1}] {cap_lp}')

# ── LSTM: Beam Search vs Greedy ───────────────────────────────────────────────
print('\n[Bonus-Beam] LSTM: Beam Search (k=5) vs Greedy')
print('='*65)
lstm_beam_results = compare_lstm_beam_vs_greedy(
    model=lstm_scratch,
    cnn_features=cnn_test[:N_BEAM],
    idx2word=idx2word,
    k=5,
    max_length=SEQ_LENGTH,
    n_samples=N_BEAM,
)

# LSTM dengan length penalty
print('\n[Bonus-Beam] LSTM: Beam Search dengan Length Penalty (alpha=0.6)')
for i, feat in enumerate(cnn_test[:3]):
    cap_lp = beam_search_lstm_with_length_penalty(
        model=lstm_scratch,
        cnn_feature=feat,
        idx2word=idx2word,
        k=5,
        max_length=SEQ_LENGTH,
        alpha=0.6,
    )
    print(f'  [{i+1}] {cap_lp}')


## Bagian 6-Bonus-2 — Batch Inference Benchmark (RNN & LSTM Scratch)

Mengukur **throughput** (gambar/detik) dan **latency** implementasi scratch
RNN dan LSTM pada berbagai ukuran batch.

In [ ]:
from rnn.bonus.bonus_batch_inference import (
    compare_batch_sizes, evaluate_batch_bleu, evaluate_batch_bleu_with_beam
)
from lstm.bonus.bonus_batch_inference import (
    compare_batch_sizes_lstm, evaluate_batch_bleu_lstm,
    evaluate_batch_bleu_lstm_with_beam
)

N_BENCH2 = min(200, len(cnn_test))
BENCH_BATCH_SIZES = (1, 8, 16, 32, 64)

# ── RNN Benchmark ──────────────────────────────────────────────────────────────
print('[Bonus-Bench] RNN Scratch — batch size benchmark:')
rnn_bench = compare_batch_sizes(
    model=rnn_scratch,
    cnn_features=cnn_test[:N_BENCH2],
    gt_captions=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    batch_sizes=BENCH_BATCH_SIZES,
    max_length=SEQ_LENGTH,
    n_samples=N_BENCH2,
    verbose=True,
)

# RNN BLEU dengan greedy
print('\n[Bonus-Bench] RNN Scratch BLEU (greedy, test subset):')
rnn_scratch_bleu = evaluate_batch_bleu(
    model=rnn_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=32,
    verbose=True,
)

# RNN BLEU dengan beam search
print('\n[Bonus-Bench] RNN Scratch BLEU (beam search k=5, test subset):')
rnn_scratch_bleu_beam = evaluate_batch_bleu_with_beam(
    model=rnn_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    k=5,
    batch_size=32,
    verbose=True,
)

# ── LSTM Benchmark ─────────────────────────────────────────────────────────────
print('\n[Bonus-Bench] LSTM Scratch — batch size benchmark:')
lstm_bench = compare_batch_sizes_lstm(
    model=lstm_scratch,
    cnn_features=cnn_test[:N_BENCH2],
    gt_captions=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    batch_sizes=BENCH_BATCH_SIZES,
    max_length=SEQ_LENGTH,
    n_samples=N_BENCH2,
    verbose=True,
)

# LSTM BLEU dengan greedy & beam
print('\n[Bonus-Bench] LSTM Scratch BLEU (greedy, test subset):')
lstm_scratch_bleu = evaluate_batch_bleu_lstm(
    model=lstm_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    batch_size=32,
    verbose=True,
)

print('\n[Bonus-Bench] LSTM Scratch BLEU (beam search k=5, test subset):')
lstm_scratch_bleu_beam = evaluate_batch_bleu_lstm_with_beam(
    model=lstm_scratch,
    cnn_features_test=cnn_test[:N_BENCH2],
    gt_captions_test=gt_captions_test[:N_BENCH2],
    idx2word=idx2word,
    max_length=SEQ_LENGTH,
    k=5,
    batch_size=32,
    verbose=True,
)


## Bagian 7 (Bonus) — Training RNN & LSTM From Scratch (NumPy BPTT)

Demonstrasi **Backpropagation Through Time (BPTT)** secara penuh menggunakan
implementasi NumPy (tanpa Keras):
1. `gradient_checker_rnn / lstm` — verifikasi gradien analitik vs numerik
2. `train_rnn_scratch / train_lstm_scratch` — training loop lengkap dengan Adam

> **Dataset**: subset kecil (200 training, 50 val) agar dapat dijalankan di notebook.
> Untuk training penuh, tingkatkan ukuran data dan jumlah epoch.

In [ ]:
from rnn.bonus.bonus_backward import (
    gradient_checker_rnn, train_one_step, train_rnn_scratch
)
from lstm.bonus.bonus_backward import (
    gradient_checker_lstm, train_one_step_lstm, train_lstm_scratch
)
from rnn.scratch.model_scratch import RNNScratch as _RNNScratch2
from lstm.scratch.model_scratch import LSTMScratch as _LSTMScratch2
import numpy as np

# ── Data subset kecil ─────────────────────────────────────────────────────────
N_SMALL = 200
N_VAL   = 50
X_small     = cnn_train[:N_SMALL]
seq_small   = seq_train[:N_SMALL]
lbl_small   = lbl_train[:N_SMALL]
X_val_sm    = cnn_val[:N_VAL]
seq_val_sm  = seq_val[:N_VAL]
lbl_val_sm  = lbl_val[:N_VAL]

# ── RNN Gradient Checker ───────────────────────────────────────────────────────
print('[Bonus-BPTT] RNN Gradient Checker (sample tunggal, eps=1e-4)...')
print('  Verifikasi: gradien analitik (backward) vs numerik (finite diff).')

_rnn_tiny = _RNNScratch2(
    vocab_size=VOCAB_SIZE, embed_dim=64, hidden_dim=64,
    num_layers=1, feature_dim=FEATURE_DIM
)
_rnn_tiny.build()

gradient_checker_rnn(
    model=_rnn_tiny,
    cnn_feature_sample=X_small[:1],
    token_sample=seq_small[:1],
    label_sample=lbl_small[:1],
    epsilon=1e-4,
    verbose=True,
)

# ── RNN Training from Scratch ─────────────────────────────────────────────────
print('\n[Bonus-BPTT] Training RNN from scratch (3 epoch, Adam, subset 200 sampel)...')
rnn_scratch_trained = train_rnn_scratch(
    model=_rnn_tiny,
    train_features=X_small,
    train_seqs=seq_small,
    train_targets=lbl_small,
    val_features=X_val_sm,
    val_seqs=seq_val_sm,
    val_targets=lbl_val_sm,
    epochs=3,
    batch_size=16,
    lr=0.001,
    optimizer='adam',
    verbose=True,
)

# ── LSTM Gradient Checker ─────────────────────────────────────────────────────
print('\n[Bonus-BPTT] LSTM Gradient Checker (sample tunggal, eps=1e-4)...')

_lstm_tiny = _LSTMScratch2(
    vocab_size=VOCAB_SIZE, embed_dim=64, hidden_dim=64,
    num_layers=1, feature_dim=FEATURE_DIM
)
_lstm_tiny.build()

gradient_checker_lstm(
    model=_lstm_tiny,
    cnn_feature_sample=X_small[:1],
    token_sample=seq_small[:1],
    label_sample=lbl_small[:1],
    epsilon=1e-4,
    verbose=True,
)

# ── LSTM Training from Scratch ────────────────────────────────────────────────
print('\n[Bonus-BPTT] Training LSTM from scratch (3 epoch, Adam, subset 200 sampel)...')
lstm_scratch_trained = train_lstm_scratch(
    model=_lstm_tiny,
    train_features=X_small,
    train_seqs=seq_small,
    train_targets=lbl_small,
    val_features=X_val_sm,
    val_seqs=seq_val_sm,
    val_targets=lbl_val_sm,
    epochs=3,
    batch_size=16,
    lr=0.001,
    optimizer='adam',
    verbose=True,
)
print('[Bonus-BPTT] Selesai.')
